# Intelligent Warehouse Robot Navigation
## D3: Implementation – Algorithms & Visualization

This notebook implements all search algorithms for the Warehouse Robot Navigation problem as defined in D1 and D2.

### Problem Summary
- Robot navigates an **N×N grid** from `(0,0)` → `(N-1, N-1)`
- Grid contains **static obstacles** and **k weighted items**
- Robot has a fixed **energy budget E₀** (each move costs 1 energy)
- **Objective**: Maximize total collected weight `W(π)` subject to `L(π) ≤ E₀`

### Algorithms Implemented
1. **BFS** – Breadth-First Search (uninformed)
2. **UCS** – Uniform Cost Search (uninformed)
3. **Greedy Best-First Search** (informed, uses h1 or h2)
4. **A\*** (informed, uses h1 or h2)
5. **Modified A\*** – Branch-and-Bound A* for maximum weight (optimal)

### Heuristics
- **h1**: Manhattan distance to nearest remaining item (or delivery)
- **h2**: h1 + MST over remaining items + distance from last item to delivery

## Importing all required libraries

In [1]:
from collections import deque
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap


## Environment – Grid, State, and Problem Definition

We define the `WarehouseEnv` class which encapsulates:
- The N×N grid with obstacles
- Item positions and weights
- State representation: `(x, y, remaining_items_frozenset, energy)`
- Transition model T(s, a)
- Visualization

In [6]:
class WarehosueEnv:
    """
    State is represented by (x, y, remaining_items_frozenset, energy)
    where (x, y) is the current position of the robot, remaining_items_frozenset is a frozenset of indices of items not yet collected, and energy is the remaining energy of the robot.
    Grid convention: (0,0) = top-left, (N-1,N-1) = bottom-right.
    Directions: Up=(-1,0), Down=(+1,0), Left=(0,-1), Right=(0,+1)
    """
    DIRECTIONS = {
        'UP':    (-1,  0),
        'DOWN':  ( 1,  0),
        'LEFT':  ( 0, -1),
        'RIGHT': ( 0,  1),
    }

    def __init__(self, N: int, energy: int, obstacle_positions: list,
                 item_positions: list, item_weights: list, seed: int = 42):
        """
        Parameters
        ----------
        N                  : grid size (N x N)
        energy             : initial energy budget E0
        obstacle_positions : list of (row, col) tuples
        item_positions     : list of (row, col) tuples (one per item)
        item_weights       : list of int weights (one per item)
        seed               : random seed (for reproducibility)
        """
        assert N >= 2, "Grid must be at least 2x2"
        assert len(item_positions) == len(item_weights), "Mismatch: positions vs weights"

        self.N = N
        self.E0 = energy
        self.obstacles = set(map(tuple, obstacle_positions))  # Creating a set for O(1) lookups
        self.item_positions = [tuple(p) for p in item_positions]   # index -> (r,c)
        self.item_weights = list(item_weights)                   # index -> weight
        self.k = len(item_positions)                             # number of items
        self.start = (0, 0)
        self.goal = (N-1, N-1)

        # Build a position -> item_index lookup for quick checks during state transitions
        self.pos_to_item = {pos: idx for idx, pos in enumerate(self.item_positions)}

        # Validate: no obstacles or items on start/goal
        assert self.start not in self.obstacles, "Start cell is blocked!"
        assert self.goal not in self.obstacles, "Goal cell is blocked!"
        for pos in self.item_positions:
            assert pos not in self.obstacles, f"Item at {pos} overlaps obstacle!"

    def initial_state(self):
        """s0 = (0, 0, frozenset(all item indices), E0)"""
        return (0, 0, frozenset(range(self.k)), self.E0)

    def is_goal(self, state) -> bool:
        x, y, _, e = state
        return (x, y) == self.goal and e >= 0

    def collected_weight(self, state) -> int:
        """Weight of items collected so far (i.e. NOT in remaining set)."""
        _, _, remaining, _ = state
        return sum(self.item_weights[i] for i in range(self.k)
                   if i not in remaining)

    # Transition model T(s, a) -> s'
    def get_successors(self, state):
        """
        Returns list of (action_name, successor_state, step_cost).

        Actions:
        Move(dir)  - cost 1, valid if adjacent cell is in-grid and not obstacle,
                    and e > 0.
        Pickup(i)  - cost 0, valid if robot is on item i and i in remaining.
                    (Auto-applied after each move in most algorithms, but
                    modeled as explicit action here for completeness.)
        """
        x, y, remaining, e = state
        successors = []

        # Move actions
        if e > 0:
            for name, (dr, dc) in self.DIRECTIONS.items():
                nx, ny = x + dr, y + dc
                if 0 <= nx < self.N and 0 <= ny < self.N:
                    if (nx, ny) not in self.obstacles:
                        new_e = e - 1
                        new_rem = remaining
                        # Auto-pickup: if item exists at new cell, collect it
                        if (nx, ny) in self.pos_to_item:
                            idx = self.pos_to_item[(nx, ny)]
                            if idx in remaining:
                                new_rem = remaining - {idx}
                        new_state = (nx, ny, new_rem, new_e)
                        successors.append((f'Move({name})', new_state, 1))

        return successors

    # Grid validation
    def is_reachable(self) -> bool:
        """BFS check: is goal reachable from start ignoring energy?"""
        visited = {self.start}
        queue = deque([self.start])
        while queue:
            x, y = queue.popleft()
            if (x, y) == self.goal:
                return True
            for dr, dc in self.DIRECTIONS.values():
                nx, ny = x + dr, y + dc
                if (0 <= nx < self.N and 0 <= ny < self.N
                        and (nx, ny) not in self.obstacles
                        and (nx, ny) not in visited):
                    visited.add((nx, ny))
                    queue.append((nx, ny))
        return False

print("WarehouseEnv defined.")

WarehouseEnv defined.


In [7]:
# TESTING
env = WarehosueEnv(
    N=4,
    energy=6,
    obstacle_positions=[(1, 1)],
    item_positions=[(0, 2), (2, 3)],
    item_weights=[5, 3],
)
s0 = env.initial_state()
print("Initial:", s0)
print("Goal?", env.is_goal(s0))
print("Collected weight:", env.collected_weight(s0))
succ = env.get_successors(s0)
print("Successors:", succ)
print("Reachable?", env.is_reachable())

Initial: (0, 0, frozenset({0, 1}), 6)
Goal? False
Collected weight: 0
Successors: [('Move(DOWN)', (1, 0, frozenset({0, 1}), 5), 1), ('Move(RIGHT)', (0, 1, frozenset({0, 1}), 5), 1)]
Reachable? True
